# Wikimedia Dump Converter

This Colab notebook installs the converter as a package and exposes a small widget UI for converting single Wikimedia dumps or dump catalog URLs.

In [ ]:
# @title
import re
import subprocess
import sys

GITHUB_REPO = "https://github.com/0xEodum/WikiParser.git"
# boto3 нужен только для загрузки в S3; в Colab включаем по умолчанию, чтобы
# storage="s3"/"both" работал без правки кода. Google Drive дополнительных
# пакетов не требует (google.colab предустановлен).
INSTALL_S3_EXTRA = True

def clean_github_repo_url(value):
    text = str(value).strip()
    match = re.search(r"https://github\.com/[^\]\)\s]+", text)
    if match:
        text = match.group(0)
    text = text.rstrip("/.)]")
    if not text.startswith("https://github.com/"):
        raise ValueError(f"Expected a GitHub HTTPS URL, got: {value!r}")
    if not text.endswith(".git"):
        text = f"{text}.git"
    return text

repo_url = clean_github_repo_url(GITHUB_REPO)
spec = f"wiki-dump-converter[s3] @ git+{repo_url}" if INSTALL_S3_EXTRA else f"git+{repo_url}"
print(f"Installing from: {spec}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec, "ipywidgets"])

In [ ]:
#@title Настройки конвертации { display-mode: "form" }
#@markdown ### Основные параметры
source = "https://dumps.wikimedia.org/other/mediawiki_content_current/enwiki/2026-05-01/xml/bzip2/" #@param {type:"string"}
output_dir = "/content/wiki_output" #@param {type:"string"}

#@markdown ---
#@markdown ### Настройки парсинга (0 = без лимита)
pages_per_file = 1 #@param {type:"integer"}
limit = 0 #@param {type:"integer"}
archive_limit = 0 #@param {type:"integer"}
namespaces = "0" #@param {type:"string"}
lead_title = "Introduction" #@param {type:"string"}
force_catalog = True #@param {type:"boolean"}
include_redirects = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### Формат и хранилище
output_format = "json" #@param ["json", "ontology"]
storage = "local" #@param ["local", "s3", "both"]

#@markdown ---
#@markdown ### Настройки S3
#@markdown Ключи берутся из **Colab Secrets** (значок 🔑 слева): `S3_BUCKET`, `S3_ENDPOINT`,
#@markdown `S3_REGION`, `S3_ACCESS_KEY`, `S3_SECRET_KEY`. Поля ниже опциональны и
#@markdown **переопределяют** значение из секретов, если заполнены.
s3_bucket = "" #@param {type:"string"}
s3_prefix = "" #@param {type:"string"}
s3_endpoint = "" #@param {type:"string"}
s3_region = "" #@param {type:"string"}
s3_access_key = "" #@param {type:"string"}
s3_secret_key = "" #@param {type:"string"}

#@markdown ---
#@markdown ### Загрузка на Google Drive (независимо от S3)
upload_to_drive = False #@param {type:"boolean"}
gdrive_dir = "/content/drive/MyDrive/wiki_output" #@param {type:"string"}
#@markdown Для ontology-формата создаётся множество мелких файлов — копирование zip-архивом на Drive в разы быстрее.
gdrive_as_zip = True #@param {type:"boolean"}

# ==========================================
# Исполняемый код начинается здесь
# ==========================================
from pathlib import Path
import json
import shutil
import time

from tqdm.auto import tqdm

from wikiparser.dump_converter import convert_catalog, convert_dump, is_catalog_source, parse_namespaces
from wikiparser.s3_io import load_s3_config, upload_directory_to_s3


def section(title: str) -> None:
    print("\n" + "=" * 60 + f"\n  {title}\n" + "=" * 60)


def optional_int(val):
    return val if val and val > 0 else None


def get_secret(name: str):
    """Достаёт секрет из Colab Secrets; вне Colab или без доступа возвращает None."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


def resolve_secret(widget_value: str, secret_name: str) -> str:
    """Виджет-значение имеет приоритет; иначе берём из секретов; иначе пусто."""
    if widget_value:
        return widget_value
    return get_secret(secret_name) or ""


def mask(value: str) -> str:
    if not value:
        return "(не задано)"
    return value[:3] + "…" if len(value) > 3 else "•••"


# --- Разрешаем настройки S3 (виджет > секреты) ---
if storage in {"s3", "both"}:
    s3_bucket = resolve_secret(s3_bucket, "S3_BUCKET")
    s3_endpoint = resolve_secret(s3_endpoint, "S3_ENDPOINT")
    s3_region = resolve_secret(s3_region, "S3_REGION")
    s3_access_key = resolve_secret(s3_access_key, "S3_ACCESS_KEY")
    s3_secret_key = resolve_secret(s3_secret_key, "S3_SECRET_KEY")

    section("Конфигурация S3")
    print(f"  bucket      : {s3_bucket or '(не задано)'}")
    print(f"  prefix      : {s3_prefix or '(пусто)'}")
    print(f"  endpoint    : {s3_endpoint or '(по умолчанию AWS)'}")
    print(f"  region      : {s3_region or '(не задано)'}")
    print(f"  access_key  : {mask(s3_access_key)}")
    print(f"  secret_key  : {mask(s3_secret_key)}")
    if not s3_bucket:
        raise ValueError(
            "Не задан S3 bucket. Добавьте секрет S3_BUCKET или заполните поле s3_bucket."
        )

# --- Конвертация с прогресс-баром ---
section(f"Конвертация (формат: {output_format}, источник: {source.rstrip('/').split('/')[-1] or source})")
page_bar = tqdm(desc="Страницы", unit=" стр")

def on_page(page):
    page_bar.update(1)
    page_bar.set_postfix_str(str(page.get("title") or "")[:40])

def on_archive(src, index, total):
    page_bar.write(f"  → Архив {index}/{total}: {src.rstrip('/').split('/')[-1] or src}")

started = time.time()
kwargs = dict(
    output_dir=output_dir,
    pages_per_file=pages_per_file,
    lead_title=lead_title,
    namespaces=parse_namespaces(namespaces),
    skip_redirects=not include_redirects,
    limit=optional_int(limit),
    output_format=output_format,
    on_page=on_page,
)

if force_catalog or is_catalog_source(source):
    summary = convert_catalog(
        catalog_url=source,
        archive_limit=optional_int(archive_limit),
        on_archive=on_archive,
        **kwargs,
    )
    page_bar.close()
    print(
        f"✓ Записано {summary.pages_written} страниц из "
        f"{summary.archives_processed}/{summary.archives_seen} архивов "
        f"за {time.time() - started:.1f} c"
    )
else:
    summary = convert_dump(source=source, **kwargs)
    page_bar.close()
    print(f"✓ Записано {summary.pages_written} страниц за {time.time() - started:.1f} c")

# --- Выгрузка в S3 с прогресс-баром ---
if storage in {"s3", "both"}:
    section("Выгрузка в S3")
    config = load_s3_config(
        bucket=s3_bucket or None,
        prefix=s3_prefix,
        endpoint_url=s3_endpoint or None,
        region_name=s3_region or None,
        access_key_id=s3_access_key or None,
        secret_access_key=s3_secret_key or None,
    )
    total_files = sum(1 for p in Path(summary.output_dir).rglob("*") if p.is_file())
    up_bar = tqdm(total=total_files, desc="Файлы → S3", unit=" файл")
    s3_summary = upload_directory_to_s3(
        summary.output_dir, config, on_file=lambda done, total, key: up_bar.update(1)
    )
    up_bar.close()
    print(f"✓ Загружено {s3_summary.files_uploaded} файлов в s3://{s3_summary.bucket}/{s3_summary.prefix}")

# --- Выгрузка на Google Drive ---
if upload_to_drive:
    section("Выгрузка на Google Drive")
    from google.colab import drive
    drive.mount("/content/drive")

    dest_dir = Path(gdrive_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)

    if gdrive_as_zip:
        print("Упаковываю в zip и копирую на Drive…")
        archive_base = str(dest_dir / Path(summary.output_dir).name)
        zip_path = shutil.make_archive(archive_base, "zip", summary.output_dir)
        print(f"✓ Архив на Google Drive: {zip_path}")
    else:
        print("Копирую каталог на Drive (может быть медленно для множества файлов)…")
        target = dest_dir / Path(summary.output_dir).name
        shutil.copytree(summary.output_dir, target, dirs_exist_ok=True)
        print(f"✓ Каталог на Google Drive: {target}")

# --- Итоговая сводка ---
section("Готово")
all_files = [p for p in Path(summary.output_dir).rglob("*") if p.is_file()]
json_files = sorted(p for p in all_files if p.suffix == ".json")
html_files = [p for p in all_files if p.suffix == ".html"]
print(f"Директория вывода : {summary.output_dir}")
print(f"Всего файлов      : {len(all_files)}  (json: {len(json_files)}, html: {len(html_files)})")

if json_files:
    sample = json.loads(json_files[0].read_text(encoding="utf-8"))
    print(f"\nПример из первого JSON ({json_files[0].name}):")
    print(json.dumps(sample[0] if isinstance(sample, list) else sample, ensure_ascii=False, indent=2)[:1200])

In [ ]:
#@title (Опционально) Скачать вывод локальным zip-архивом
import shutil

zip_path = shutil.make_archive("/content/wiki_output", "zip", output_dir)
print(f"Создан архив: {zip_path}")

# В Colab можно скачать его в браузер:
try:
    from google.colab import files
    files.download(zip_path)
except Exception as exc:
    print(f"Автоскачивание недоступно ({exc}); файл лежит здесь: {zip_path}")